## pydantic 使用

In [11]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

load_dotenv(override=True)

model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"}
)

In [12]:
## 格式化输出

class Person(BaseModel):
    """
    任务信息
    """
    name: str=Field(description="姓名")
    age: int=Field(description="年龄")
    occupation: str=Field(description="职位")

model1 = model.with_structured_output(schema=Person)
response = model1.invoke("小许是一个Java开发工程师")
print(response)

name='小许' age=0 occupation='Java开发工程师'


#### 举例 2

In [13]:
class Movie(BaseModel):
    """
    电影的详细信息
    """
    title:str=Field(description="电影的标题")
    director:str=Field(description="导演")
    year:int=Field(description="发布年份")
    rating:float=Field(description="评分")

model2 = model.with_structured_output(schema=Movie)
response2 = model2.invoke("帮我找一下环太平洋电影的信息")
print(response2)

title='环太平洋' director='吉尔莫·德尔·托罗' year=2013 rating=7.0


## 高级特性

#### 可选字段

In [14]:
class Person(BaseModel):
    """
    任务信息
    """
    name: str=Field(description="姓名")
    age: Optional[int]=Field(description="年龄")
    occupation: str=Field(description="职位")

model1 = model.with_structured_output(schema=Person)
response = model1.invoke("小许是一个Java开发工程师")
print(response)

name='小许' age=None occupation='Java开发工程师'


#### 默认值

In [15]:
## 默认值：当模型没有给出某个字段时，使用预设的默认值填充

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(default=18, description="年龄")        # 带默认值的字段是「可选」的
    occupation: str = Field(default="未知", description="职位")

model_default = model.with_structured_output(schema=Person)
print(model_default.invoke("小许是一个Java开发工程师"))


name='小许' age=18 occupation='Java开发工程师'


#### 枚举 / 字面量 Literal

In [16]:
## 字面量（Literal）：把字段的取值范围限定为几个固定值，模型只能从中选一个
from typing import Literal

class Resume(BaseModel):
    """简历信息"""
    name: str = Field(description="姓名")
    level: Literal["初级", "中级", "高级", "专家"] = Field(description="职级，只能取这四者之一")
    city: str = Field(description="所在城市")

model_literal = model.with_structured_output(schema=Resume)
print(model_literal.invoke("张三是高级Java开发工程师，base 在上海"))


name='张三' level='高级' city='上海'


#### 枚举类型 Enum

In [17]:
## 枚举（Enum）：比 Literal 更规范的可选值定义，可以附带更多语义
from enum import Enum

class Sentiment(str, Enum):
    """情感倾向的三种取值，值即为中文标签"""
    positive = "正面"
    negative = "负面"
    neutral = "中性"

class Review(BaseModel):
    """影评分析"""
    content: str = Field(description="评论原文")
    sentiment: Sentiment = Field(description="情感倾向")

model_enum = model.with_structured_output(schema=Review)
print(model_enum.invoke("这部电影太烂了，浪费我两个小时"))


content='这部电影太烂了，浪费我两个小时' sentiment=<Sentiment.negative: '负面'>


#### 列表字段

In [18]:
## 列表字段：让模型一次性输出多个同类型元素，而不是拼在一个字符串里
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    actors: list[str] = Field(description="主演列表")
    genres: list[str] = Field(description="类型标签列表")

model_list = model.with_structured_output(schema=Movie)
print(model_list.invoke("请介绍电影《流浪地球》的主演和类型"))


title='流浪地球' actors=['吴京', '屈楚萧', '李光洁', '吴孟达', '赵今麦'] genres=['科幻', '灾难', '冒险']


#### 嵌套模型

In [19]:
## 嵌套模型：一个模型可以引用另一个模型作为字段，表达多层级的结构
class Address(BaseModel):
    """住址信息"""
    city: str = Field(description="城市")
    street: str = Field(description="街道")

class Employee(BaseModel):
    """员工信息"""
    name: str = Field(description="姓名")
    address: Address = Field(description="家庭住址")   # 字段类型是另一个 Pydantic 模型

model_nested = model.with_structured_output(schema=Employee)
print(model_nested.invoke("小许住在北京市朝阳区望京街道"))


name='小许' address=Address(city='北京市', street='朝阳区望京街道')


#### 字段校验

In [20]:
## 字段校验：用 Field 的内置约束限制取值范围，再用 field_validator 做自定义校验
from pydantic import field_validator, ValidationError

class Product(BaseModel):
    """商品信息"""
    name: str = Field(description="商品名称", min_length=2, max_length=20)  # 长度必须在 2~20 之间
    price: float = Field(description="价格", gt=0)                            # 必须大于 0
    stock: int = Field(description="库存", ge=0)                              # 必须大于等于 0

    @field_validator("name")
    @classmethod
    def name_not_blank(cls, v: str) -> str:
        """自定义校验：名称不能只包含空白字符，并自动去掉首尾空格"""
        if not v.strip():
            raise ValueError("商品名称不能为空")
        return v.strip()

# 手动构造一个不合法实例，观察校验报错
try:
    Product(name="  ", price=-1, stock=5)
except ValidationError as e:
    print("校验失败：", e)


校验失败： 2 validation errors for Product
name
  Value error, 商品名称不能为空 [type=value_error, input_value='  ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
price
  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than


#### 别名 alias

In [21]:
## 别名（alias）：让 JSON 输出使用英文 key，而 Python 属性保持可读性
class Order(BaseModel):
    """订单信息"""
    order_id: str = Field(alias="orderId", description="订单号")
    total_amount: float = Field(alias="totalAmount", description="订单总金额")

model_alias = model.with_structured_output(schema=Order)
print(model_alias.invoke("订单号 A1001，总金额 99.9 元"))


order_id='A1001' total_amount=99.9


#### 模型配置 ConfigDict

In [22]:
## 模型配置（ConfigDict）：控制模型的整体行为
from pydantic import ConfigDict

class ConfigDemo(BaseModel):
    """演示模型配置"""
    model_config = ConfigDict(
        extra="ignore",   # 遇到 schema 里没有的字段时直接忽略，而不是报错
        frozen=True,      # 实例创建后不可修改（不可变对象）
    )
    name: str = Field(description="姓名")

obj = ConfigDemo(name="小许")
# obj.name = "别人"   # frozen=True 时，这行会抛出 ValidationError
print(obj)


name='小许'


#### 联合类型 Union

In [23]:
## 联合类型（Union）：字段可以接受多种类型中的任意一种
from typing import Union

class Contact(BaseModel):
    """联系方式"""
    name: str = Field(description="姓名")
    phone: Union[str, int] = Field(description="电话号码，可能是字符串或数字")
    email: Union[str, None] = Field(default=None, description="邮箱，可能缺失")

model_union = model.with_structured_output(schema=Contact)
print(model_union.invoke("小许，电话 13812345678，邮箱 xiaoxu@example.com"))


name='小许' phone='13812345678' email='xiaoxu@example.com'
